In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:


class MultiAgentTransformer(nn.Module):
    def __init__(self, num_agents, state_dim, pos_dim, site_pos_dim, site_quality_dim, d_model, num_heads, num_layers):
        super(MultiAgentTransformer, self).__init__()
        
        # Embeddings
        self.agent_state_embedding = nn.Linear(state_dim, d_model)
        self.agent_pos_embedding = nn.Linear(pos_dim, d_model)
        self.site_pos_embedding = nn.Linear(site_pos_dim, d_model)
        self.site_quality_embedding = nn.Linear(site_quality_dim, d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layers
        self.success_fc = nn.Linear(d_model, 1)
        self.time_fc = nn.Linear(d_model, 1)
        
    def forward(self, agent_states, agent_positions, site_positions, site_qualities):
        # Embeddings
        agent_state_embed = self.agent_state_embedding(agent_states)
        agent_pos_embed = self.agent_pos_embedding(agent_positions)
        site_pos_embed = self.site_pos_embedding(site_positions)
        site_quality_embed = self.site_quality_embedding(site_qualities)
        
        # Combine embeddings
        combined_embed = agent_state_embed + agent_pos_embed + site_pos_embed + site_quality_embed
        
        # Transformer Encoder
        transformer_output = self.transformer_encoder(combined_embed)
        
        # Output layers
        success_pred = torch.sigmoid(self.success_fc(transformer_output))
        time_pred = F.relu(self.time_fc(transformer_output))
        
        return success_pred, time_pred

In [ ]:

# Example Data (Replace with your actual data)
num_agents = 100
num_sites = 4
state_dim = 5
pos_dim = 3
site_pos_dim = 3
site_quality_dim = 2
num_samples = 1000

agent_states = torch.rand(num_samples, num_agents, state_dim)
agent_positions = torch.rand(num_samples, num_agents, pos_dim)
site_positions = torch.rand(num_samples, num_sites, site_pos_dim)
site_qualities = torch.rand(num_samples, num_sites, site_quality_dim)
success_labels = torch.randint(0, 2, (num_samples, 1)).float()
time_labels = torch.rand(num_samples, 1)

# Create a dataset and data loaders
dataset = TensorDataset(agent_states, agent_positions, site_positions, site_qualities, success_labels, time_labels)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model, Loss, and Optimizer
model = MultiAgentTransformer(
    num_agents=num_agents,
    state_dim=state_dim,
    pos_dim=pos_dim,
    site_pos_dim=site_pos_dim,
    site_quality_dim=site_quality_dim,
    d_model=64,
    num_heads=8,
    num_layers=6
)

criterion_success = nn.BCELoss()
criterion_time = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for agent_states, agent_positions, site_positions, site_qualities, success_labels, time_labels in train_loader:
        optimizer.zero_grad()
        success_pred, time_pred = model(agent_states, agent_positions, site_positions, site_qualities)
        
        loss_success = criterion_success(success_pred, success_labels)
        loss_time = criterion_time(time_pred, time_labels)
        loss = loss_success + loss_time
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

# Testing Loop
model.eval()
test_loss = 0.0
with torch.no_grad():
    for agent_states, agent_positions, site_positions, site_qualities, success_labels, time_labels in test_loader:
        success_pred, time_pred = model(agent_states, agent_positions, site_positions, site_qualities)
        
        loss_success = criterion_success(success_pred, success_labels)
        loss_time = criterion_time(time_pred, time_labels)
        loss = loss_success + loss_time
        
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader)}")

# Example usage after training
sample_index = 0  # Example sample index
sample_agent_states = agent_states[sample_index].unsqueeze(0)
sample_agent_positions = agent_positions[sample_index].unsqueeze(0)
sample_site_positions = site_positions[sample_index].unsqueeze(0)
sample_site_qualities = site_qualities[sample_index].unsqueeze(0)

model.eval()
with torch.no_grad():
    success_pred, time_pred = model(sample_agent_states, sample_agent_positions, sample_site_positions, sample_site_qualities)
    print(f"Predicted Success: {success_pred.item()}, Predicted Time: {time_pred.item()}")
